In [2]:
from pyspark.sql import SparkSession

# Initializing Spark Session
spark = SparkSession.builder.appName("Week6_Assignment").getOrCreate()

# Loading the dataset
df_source = spark.read.csv("week-6_dataset.csv", header=True, inferSchema=True)

# Creating a parquet version
df_source.write.mode('overwrite').parquet("dataset_parquet")

print("Setup Complete! Dataset loaded.")
df_source.show(5)

Setup Complete! Dataset loaded.
+-------+----------+---------+-------+--------------------+---------+-------+----------+-------+--------+
|user_id|product_id| category|  price|            old_name|   status| amount|base_price| region|priority|
+-------+----------+---------+-------+--------------------+---------+-------+----------+-------+--------+
|   8621|       713|     Toys|1022.04|   Sports Monitor v5|  Pending| 5071.2|   1014.24|Midwest|  Medium|
|   7024|       720|   Sports|  125.7|    Clothing Tool v3|Completed| 156.52|     78.26|Midwest|    High|
|   5226|       818|Furniture|1758.51|Appliances System v5|  Pending|3421.88|   1710.94|   West|  Medium|
|   NULL|       679|   Sports| 847.74| Furniture System v3| Canceled|3998.85|    799.77|  South|    High|
|   9454|       355|   Sports|1314.07|    Sports Widget v1| Canceled|2601.32|   1300.66|  North|    High|
+-------+----------+---------+-------+--------------------+---------+-------+----------+-------+--------+
only showing t

**Implementation Logic:** We use inferSchema=True so Spark automatically figures out if a column is a number or text, saving us time.

In [3]:
df_q3 = spark.read.csv("week-6_dataset.csv", header=True, inferSchema=True)

df_q3.printSchema()
df_q3.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+-------+----------+---------+-------+--------------------+---------+-------+----------+-------+--------+
|user_id|product_id| category|  price|            old_name|   status| amount|base_price| region|priority|
+-------+----------+---------+-------+--------------------+---------+-------+----------+-------+--------+
|   8621|       713|     Toys|1022.04|   Sports Monitor v5|  Pending| 5071.2|   1014.24|Midwest|  Medium|
|   7024|       720|   Sports|  125.7|    Clothing Tool v3|Completed| 156.52|     78.26|Midwest|    High|
|   5226|       818|Furniture|1758.51|Appliances System v5|  Pending

**Implementation Logic:** We filter the data first, and then select the columns. This ensures we only process the exact rows we need.

In [4]:
from pyspark.sql.functions import col

df_electronics = (df_source
    .filter(col("category") == "Electronics")
    .select("product_id", "price")
)

df_electronics.show(5)

+----------+-------+
|product_id|  price|
+----------+-------+
|       376| 261.08|
|       847| 678.07|
|       472| 465.75|
|       815|1611.79|
|       177|1867.66|
+----------+-------+
only showing top 5 rows


**Implementation Logic:** We chain our methods together to rename the column and fix the data type (casting) in one smooth step.

In [5]:
from pyspark.sql.types import DoubleType

df_revised = (df_source
    .withColumnRenamed("old_name", "new_name")
    .withColumn("price", col("price").cast(DoubleType()))
)

df_revised.select("new_name", "price").printSchema()
df_revised.select("new_name", "price").show(5)

root
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)

+--------------------+-------+
|            new_name|  price|
+--------------------+-------+
|   Sports Monitor v5|1022.04|
|    Clothing Tool v3|  125.7|
|Appliances System v5|1758.51|
| Furniture System v3| 847.74|
|    Sports Widget v1|1314.07|
+--------------------+-------+
only showing top 5 rows


**Implementation Logic:** We use the & symbol and put parentheses around each condition so PySpark understands the exact logic

In [6]:
df_high_value_completed = df_source.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

df_high_value_completed.select("product_id", "status", "amount").show(5)

+----------+---------+-------+
|product_id|   status| amount|
+----------+---------+-------+
|       134|Completed|7020.25|
|       263|Completed|2920.65|
|       752|Completed|1314.82|
|       472|Completed|1686.56|
|       177|Completed|9232.45|
+----------+---------+-------+
only showing top 5 rows


**Implementation Logic:** We use .withColumn() to easily do math on an existing column and store the answer in a brand new column.

In [7]:
df_taxed = df_source.withColumn("final_price", col("base_price") * 1.18)

df_taxed.select("product_id", "base_price", "final_price").show(5)

+----------+----------+-----------------+
|product_id|base_price|      final_price|
+----------+----------+-----------------+
|       713|   1014.24|        1196.8032|
|       720|     78.26|          92.3468|
|       818|   1710.94|        2018.9092|
|       679|    799.77|943.7285999999999|
|       355|   1300.66|        1534.7788|
+----------+----------+-----------------+
only showing top 5 rows


**Implementation Logic:** This is a mini data pipeline. It reads fast Parquet data, cleans out missing user IDs, and saves it to a simple CSV.

In [ ]:
(spark.read.parquet("dataset_parquet")
    .filter(col("user_id").isNotNull())
    .write.mode("overwrite").csv("output_cleaned_csv", header=True)
)

spark.read.csv("output_cleaned_csv", header=True).select("user_id", "category").show(5)

+-------+---------+
|user_id| category|
+-------+---------+
|   8621|     Toys|
|   7024|   Sports|
|   5226|Furniture|
|   9454|   Sports|
|   9098| Clothing|
+-------+---------+
only showing top 5 rows


**Implementation Logic:** We use the pipe symbol | to represent "OR" in PySpark so we can capture rows that match either of the two conditions.

In [9]:
df_filtered = df_source.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

df_filtered.select("product_id", "region", "priority").show(5)

+----------+-------+--------+
|product_id| region|priority|
+----------+-------+--------+
|       720|Midwest|    High|
|       679|  South|    High|
|       355|  North|    High|
|       403|  North|Critical|
|       376|  South|    High|
+----------+-------+--------+
only showing top 5 rows
